In [ ]:
# Google Colab Setup — run this cell first if on Colab
import os

if "COLAB_RELEASE_TAG" in os.environ:
    !git clone https://github.com/akhilesh-yadav/lead-scoring.git /content/lead-scoring
    %cd /content/lead-scoring
    !pip install -q -r requirements.txt
    !pip install -q -e .
    !python -m lead_scorer generate
    import sys
    sys.path.insert(0, "/content/lead-scoring")
else:
    import sys
    sys.path.insert(0, "..")

# Pipeline Walkthrough

This notebook demonstrates the 4-layer scoring pipeline step-by-step,
showing how each layer transforms the data and how to inspect intermediate results.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

DATA_ROOT = "/content/lead-scoring" if "COLAB_RELEASE_TAG" in os.environ else ".."

## Layer 1: Cleaning & Entity Resolution

Flags exclusions, DQ issues, and attempts entity resolution.

In [ ]:
from src.pipeline.stages.clean import clean_data

result = clean_data(f'{DATA_ROOT}/data/raw')

print("\n=== Cleaning Stats ===")
for k, v in result.resolution_stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Inspect exclusion flags
print("\n=== Lead Exclusions ===")
excl_cols = [c for c in result.leads.columns if c.startswith('exclude_')]
for col in excl_cols:
    print(f"  {col}: {result.leads[col].sum()}")

print(f"\n  Total excluded leads: {result.leads['is_excluded'].sum()} / {len(result.leads)}")

In [ ]:
# Inspect DQ flags
print("=== DQ Issue Distribution (Leads) ===")
dq_cols = [c for c in result.leads.columns if c.startswith('dq_') and c != 'dq_issue_count']
for col in dq_cols:
    print(f"  {col}: {result.leads[col].sum()}")

print(f"\n  Records with 0 DQ issues: {(result.leads['dq_issue_count'] == 0).sum()}")
print(f"  Records with 1+ DQ issues: {(result.leads['dq_issue_count'] > 0).sum()}")
print(f"  Records with 3+ DQ issues: {(result.leads['dq_issue_count'] >= 3).sum()}")

## Layer 2: Feature Engineering

Computes engagement, profile, account, and momentum features for every entity.

In [ ]:
from src.pipeline.stages.features import engineer_features, compute_engagement_features, compute_profile_features

features_df = engineer_features(
    result.accounts, result.leads, result.contacts, result.campaign_members
)

print(f"\nFeature columns: {len(features_df.columns)}")
print(features_df[['entity_id', 'entity_type', 'real_engagements', 
                    'days_since_last_engagement', 'level_score', 'persona_score',
                    'has_account', 'momentum_score']].head(10))

In [ ]:
# Inspect a single record's features in detail
sample_id = result.leads.iloc[0]['lead_id']
print(f"Detailed features for {sample_id}:")
eng = compute_engagement_features(sample_id, 'lead', result.campaign_members)
print(f"  Engagement: {eng.to_dict()}")

prof = compute_profile_features(
    result.leads.iloc[0].get('job_level'),
    result.leads.iloc[0].get('job_persona'),
    result.leads.iloc[0].get('title'),
)
print(f"  Profile: {prof.to_dict()}")

## Layer 3: Component Scoring

Applies scoring functions to features. Each component yields 0-100.

In [ ]:
from src.pipeline.stages.score import compute_scores, ScoringWeights

scored_df = compute_scores(features_df)

print("\n=== Score Component Statistics ===")
score_cols = ['score_engagement', 'score_profile', 'score_account', 'score_momentum', 'readiness_score']
print(scored_df[score_cols].describe().round(1))

In [ ]:
# Try different weights
print("\n=== Sensitivity: Engagement-Heavy Weights ===")
heavy_eng = compute_scores(features_df, ScoringWeights(engagement=0.6, profile=0.15, account=0.15, momentum=0.1))
print(f"  Mean readiness: {heavy_eng['readiness_score'].mean():.1f}")
print(f"  Hot records (>70): {(heavy_eng['readiness_score'] >= 70).sum()}")

print("\n=== Sensitivity: Profile-Heavy Weights ===")
heavy_prof = compute_scores(features_df, ScoringWeights(engagement=0.25, profile=0.45, account=0.2, momentum=0.1))
print(f"  Mean readiness: {heavy_prof['readiness_score'].mean():.1f}")
print(f"  Hot records (>70): {(heavy_prof['readiness_score'] >= 70).sum()}")

## Layer 4: Ranking & Tiering

Assigns tiers and produces the final ranked list.

In [ ]:
from src.pipeline.stages.rank import rank_and_tier, TierConfig

final_df = rank_and_tier(scored_df, result.leads, result.contacts, result.accounts)

print("\n=== Top 10 Records ===")
print(final_df[['rank', 'entity_id', 'entity_type', 'tier', 'readiness_score',
                'score_engagement', 'score_profile', 'score_account']].head(10).to_string())

In [ ]:
# Entity-type fairness check
print("\n=== Entity Type Fairness ===")
scorable = final_df[~final_df['is_excluded']]
fairness = scorable.groupby('entity_type').agg(
    count=('readiness_score', 'count'),
    mean=('readiness_score', 'mean'),
    median=('readiness_score', 'median'),
    pct_hot=('tier', lambda x: (x == 'Hot').mean() * 100),
).round(1)
print(fairness)

In [ ]:
# Custom tier thresholds
print("\n=== Custom Tier Config (stricter) ===")
strict = rank_and_tier(
    scored_df, result.leads, result.contacts, result.accounts,
    tier_config=TierConfig(hot_threshold=80, warm_threshold=55, nurture_threshold=25)
)
print(strict['tier'].value_counts())

## Running Tests

All pipeline functions are independently testable. Run from terminal:

```bash
cd /path/to/lead-scoring-poc
python -m pytest tests/ -v
```